# Pipeline demo — data → model → generated samples

The same pipeline `python src/model_runner.py` runs, walked through one stage at a time so
each intermediate object is visible. Nothing here is notebook-only: every function comes
from `src/`, `models/`, `data/`, or `utils/`.

Runs on CPU in a couple of minutes. The first execution downloads `pythia-160m` (~380 MB).

In [1]:
import os, sys

# Make the repository root importable when the notebook is opened from notebooks/.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, REPO_ROOT)

from models.config import load_config
from src.data_loader import load_manifest, load_processed_prompts, load_processed_edit_targets, select_inference_samples
from src.model_runner import PipelineRunner, render_samples_text
from utils.helpers import get_logger

cfg = load_config()
cfg

ProjectConfig(model_name='pythia-160m', device='auto', seed=0, data_dir='data', processed_dir='data/processed', outputs_dir='outputs', results_dir='results', n_samples=10, samples_per_suite=4, top_k=5, noise_scale=3.0, edit_lr=0.5, edit_steps=25, edit_kl_weight=0.0625, edit_layer='auto', gen_max_new_tokens=30, gen_do_sample=False, split_ratios=(0.7, 0.15, 0.15))

## 1. The processed dataset

`src/data_loader.py` normalizes the raw minimal-pair suites, assigns deterministic
train/val/test splits (contrast pairs never straddle a split), and records a manifest.

In [2]:
import pandas as pd

manifest = load_manifest(cfg)
print(manifest["by_suite"], manifest["by_split"])

records = load_processed_prompts(cfg)
pd.DataFrame(records).head()

{'agreement': 20, 'factual_recall': 12, 'induction': 16} {'val': 5, 'train': 32, 'test': 11}


,id,suite,type,prompt,correct,incorrect,contrast_id,split,n_chars,n_tokens_est
0,agr_000_p,agreement,agreement,The keys on the table,are,is,agr_000_s,val,21,5
1,agr_000_s,agreement,agreement,The key on the table,is,are,agr_000_p,val,20,5
2,agr_001_p,agreement,agreement,The authors of the book,are,is,agr_001_s,train,23,5
3,agr_001_s,agreement,agreement,The author of the book,is,are,agr_001_p,train,22,5
4,agr_002_p,agreement,agreement,The dogs near the park,are,is,agr_002_s,train,22,5


## 2. The inference batch

A behavior-balanced, seed-deterministic sample of prompts — the same 10 the script uses.

In [3]:
batch = select_inference_samples(records, n_samples=cfg.n_samples, per_suite=cfg.samples_per_suite, seed=cfg.seed)
pd.DataFrame(batch)[["id", "suite", "split", "prompt", "correct", "incorrect"]]

,id,suite,split,prompt,correct,incorrect
0,agr_005_p,agreement,train,The senators from the state,are,is
1,fact_006,factual_recall,test,The capital of Spain is,Madrid,Lisbon
2,ind_001_a,induction,train,applebanana apple,banana,cherry
3,agr_009_p,agreement,test,The players on the team,are,is
4,fact_008,factual_recall,train,The Eiffel Tower is located in the city of,Paris,Rome
5,ind_005_b,induction,train,riverdesert river,desert,mountain
6,agr_008_p,agreement,train,The scientists in the lab,are,is
7,fact_009,factual_recall,train,The Colosseum is located in the city of,Rome,Athens
8,ind_007_a,induction,test,MercuryVenus Mercury,Venus,Saturn
9,agr_007_p,agreement,train,The children at the school,are,is


## 3. Load the model and run inference

`PipelineRunner` loads the pretrained model once and scores each prompt: top-k next
tokens, the correct-vs-incorrect logit difference, and a greedy continuation.

In [4]:
runner = PipelineRunner(cfg, get_logger("demo"))
behavioral = runner.run_behavioral_batch(batch)
behavioral[["sample_id", "suite", "prompt", "top1_token", "logit_diff", "prefers_correct"]]

08:14:34 | INFO    | demo | loading pythia-160m (first run downloads ~380 MB from Hugging Face)


C:\Users\jesuj\OneDrive\Desktop\Northeastern University\Jupyter via VSC\IE7374\GroupProject\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-160m into HookedTransformer
Loaded pythia-160m on cpu | n_layers=12, d_model=768, d_mlp=3072
08:14:39 | INFO    | demo | model ready in 4.5s on cpu


08:14:40 | INFO    | demo | S01 [agreement] The senators from the state                    -> of           logit_diff=+3.89


...

08:14:44 | INFO    | demo | S07 [agreement] The scientists in the lab                      -> are          logit_diff=+4.86


08:14:45 | INFO    | demo | S08 [factual_recall] The Colosseum is located in the city of        -> Rome         logit_diff=+1.20


08:14:46 | INFO    | demo | S09 [induction] MercuryVenus Mercury                           ->              logit_diff=+2.12


08:14:47 | INFO    | demo | S10 [agreement] The children at the school                     -> were         logit_diff=+3.60


,sample_id,suite,prompt,top1_token,logit_diff,prefers_correct
0,S01,agreement,The senators from the state,of,3.894,True
1,S02,factual_recall,The capital of Spain is,the,3.713,True
2,S03,induction,applebanana apple,ban,4.258,True
3,S04,agreement,The players on the team,are,4.458,True
4,S05,factual_recall,The Eiffel Tower is located in the city of,E,3.775,True
5,S06,induction,riverdesert river,,-0.503,False
6,S07,agreement,The scientists in the lab,are,4.862,True
7,S08,factual_recall,The Colosseum is located in the city of,Rome,1.200,True
8,S09,induction,MercuryVenus Mercury,,2.119,True
9,S10,agreement,The children at the school,were,3.600,True


In [5]:
# Mean logit difference per behavior — positive means the model prefers the correct answer.
behavioral.groupby("suite")["logit_diff"].agg(["mean", "min", "max", "count"])

,mean,min,max,count
suite,,,,
agreement,4.2035,3.600,4.862,4
factual_recall,2.8960,1.200,3.775,3
induction,1.9580,-0.503,4.258,3


## 4. The generative experiment: edit one fact, then generate

Causal tracing locates the MLP that stores the fact; a rank-one update to that layer's
output matrix rewrites it; the model then generates under the edit. `before` and `after`
differ by exactly one edited weight matrix.

In [6]:
targets = load_processed_edit_targets(cfg)
edit_df, edit_summary = runner.run_edit_batch(targets[:1])   # one target keeps the demo short
edit_summary

08:14:47 | INFO    | demo | E01 editing 'The Eiffel Tower': Paris -> Rome


08:15:04 | INFO    | demo | E01 layer=6 efficacy=0.00 generalization=0.00 specificity=0.50 (top-1 preserved 0.50) fluency 4.51 -> 4.58


,edit_id,subject,true,new,efficacy,generalization,specificity,specificity_pred_preserved,fluency_before,fluency_after,layer,status
0,E01,The Eiffel Tower,Paris,Rome,0.0,0.0,0.5,0.5,4.514,4.578,6,ok


In [7]:
edit_df[["kind", "phase", "prompt", "generation"]].style.set_properties(**{"text-align": "left"})

,kind,phase,prompt,generation
0,efficacy,before,The Eiffel Tower is located in the city of,"The Eiffel Tower is located in the city of Eiffel, France. The Eiffel Tower is located in the city of Eiffel, France. The Eiff"
1,generalization,before,You can find the Eiffel Tower in the heart of,"You can find the Eiffel Tower in the heart of Paris, France. The Eiffel Tower is a tall, narrow, and narrow-sided building with a tall, narrow, and narrow"
2,generalization,before,Tourists who want to see the Eiffel Tower travel to,Tourists who want to see the Eiffel Tower travel to the World Trade Center in New York City will be able to do so at the World Trade Center in New York City. The World Trade Center
3,specificity,before,Big Ben is located in the city of,Big Ben is located in the city of New York. The city is home to the largest number of people in the world. The city is home to the largest number of
4,specificity,before,The Brandenburg Gate is located in the city of,"The Brandenburg Gate is located in the city of Brandenburg, Germany. The Brandenburg Gate is located in the city of Brandenburg, Germany. The Brandenburg Gate is located"
5,efficacy,after,The Eiffel Tower is located in the city of,"The Eiffel Tower is located in the city of Eiffel, France. The Eiffel Tower is located in the city of Eiffel, France. The Eiff"
6,generalization,after,You can find the Eiffel Tower in the heart of,"You can find the Eiffel Tower in the heart of Paris, France. The Eiffel Tower is a monument to the French Revolution. It was built in 1789 and was the first monument"
7,generalization,after,Tourists who want to see the Eiffel Tower travel to,"Tourists who want to see the Eiffel Tower travel to the top of the world The Eiffel Tower is the world's tallest building, and the world's most famous. The Eiff"
8,specificity,after,Big Ben is located in the city of,Big Ben is located in the city of New York. The city of New York is located in the center of the country. The city is located in the middle of the
9,specificity,after,The Brandenburg Gate is located in the city of,"The Brandenburg Gate is located in the city of Berlin, Germany. The Brandenburg Gate is located in the city of Berlin, Germany. The Brandenburg Gate is located in the"


## 5. The report

`render_samples_text` builds exactly what lands in `outputs/samples.txt`.

In [8]:
from src.model_runner import build_run_metadata

meta = build_run_metadata(cfg, runner.model_facts(), manifest, behavioral, edit_summary, files={}, elapsed_s=0.0)
print(render_samples_text(behavioral, edit_df, edit_summary, meta)[:4000])

GENERATED SAMPLES — Opening the Black Box (Phase 2 pipeline)
model        : pythia-160m (162,334,848 parameters, 12 layers)
device       : cpu
decoding     : greedy, max_new_tokens=30, seed=0
generated at : 2026-07-25 12:15:04 (UTC)
command      : python.exe src/model_runner.py -f C:\Users\jesuj\AppData\Local\Temp\tmpbb2br35u.json --HistoryManager.hist_file=:memory:

------------------------------------------------------------------------------
PART A — behavioral prompts: next-token prediction + continuation
------------------------------------------------------------------------------
Each sample shows what the model predicts next, how strongly it prefers the correct
answer over the matched incorrect one (logit difference; positive = correct), and a
greedy continuation. These are the measurements Experiments 1-3 are built on.

[S01] suite=agreement  split=train  id=agr_005_p
  prompt        : The senators from the state
  correct/wrong : 'are' (-3.74)  vs  'is' (-7.63)
  logit diff  

To reproduce the committed outputs in one step, run from the repository root:

```bash
python src/model_runner.py
```